# Notebook 2 · Análisis Exploratorio de Datos (EDA)
**Consumo de energía eléctrica – Electro Puno S.A.A.**

Objetivos:
1. Describir los atributos (estadística descriptiva univariada y bivariada).
2. Analizar la composición por **tarifa**, **territorio** y **tiempo**.
3. Medir la **concentración** del consumo (Pareto, curva de Lorenz, Gini).
4. Construir **indicadores complementarios** a nivel distrital.
5. Segmentar los distritos con **clustering (K-Means)**.

Entrada: `data/processed/consumo_limpio.parquet` (generado en el Notebook 1).

## 1. Carga

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.float_format", "{:,.2f}".format)
OUT = Path("data/processed")

df = pd.read_parquet(OUT / "consumo_limpio.parquet")
geo = json.load(open(OUT / "puno_distritos.geojson", encoding="utf-8"))
n_periodos = df["periodo_cod"].nunique()
print(df.shape, "| periodos:", sorted(df["periodo_cod"].unique()))

(27875, 29) | periodos: [np.int64(202301), np.int64(202302), np.int64(202303), np.int64(202304), np.int64(202305), np.int64(202306), np.int64(202307), np.int64(202308), np.int64(202309), np.int64(202310), np.int64(202311), np.int64(202312), np.int64(202401), np.int64(202402), np.int64(202403), np.int64(202404), np.int64(202405), np.int64(202407), np.int64(202408), np.int64(202409), np.int64(202410), np.int64(202411), np.int64(202412), np.int64(202501), np.int64(202503), np.int64(202504), np.int64(202505), np.int64(202506), np.int64(202507), np.int64(202508), np.int64(202509), np.int64(202510), np.int64(202511), np.int64(202512), np.int64(202601), np.int64(202602), np.int64(202603), np.int64(202604), np.int64(202605), np.int64(202606), np.int64(202607), np.int64(202608)]


## 2. Visión general (KPIs)

In [2]:
bt = df[df["nivel_tension"] == "BT"]
kpi = {
    "Clientes (prom. mensual)": df["clientes"].sum() / n_periodos,
    "Energía activa (MWh/mes)": df["consumo_kwh"].sum() / 1000 / n_periodos,
    "Facturación (S/ miles/mes)": df["monto_soles"].sum() / 1000 / n_periodos,
    "kWh por cliente BT": bt["consumo_kwh"].sum() / bt["clientes"].sum(),
    "S/ por cliente BT": bt["monto_soles"].sum() / bt["clientes"].sum(),
    "Precio medio BT (S/ por kWh)": bt["monto_soles"].sum() / bt["consumo_kwh"].sum(),
    "Provincias": df["provincia"].nunique(),
    "Distritos": df["ubigeo"].nunique(),
}
pd.Series(kpi).to_frame("valor")

,valor
Clientes (prom. mensual),"349,392.83"
Energía activa (MWh/mes),"21,649.11"
Facturación (S/ miles/mes),"28,704.12"
kWh por cliente BT,52.79
S/ por cliente BT,57.99
Precio medio BT (S/ por kWh),1.10
Provincias,13.00
Distritos,109.00


## 3. Estadística descriptiva
### 3.1 Variables numéricas por nivel de tensión

In [3]:
num = ["clientes", "consumo_kwh", "monto_soles", "kwh_cliente", "soles_cliente", "precio_kwh"]
df.groupby("nivel_tension")[num].describe().T

nivel_tension                  BT           MT
clientes      count     18,687.00     9,188.00
              mean         782.78         5.09
              std        5,924.71        12.61
              min            1.00         1.00
              25%            1.00         1.00
              50%            3.00         2.00
              75%           90.00         4.00
              max      113,524.00       141.00
consumo_kwh   count     18,687.00     9,188.00
              mean      41,321.77    14,919.77
              std      415,217.27    78,133.66
              min            0.00         0.00
              25%           17.00         0.00
              50%          226.00         0.00
              75%        7,714.50     1,728.42
              max    9,149,561.60 1,141,033.49
monto_soles   count     18,687.00     9,188.00
              mean      45,392.64    38,889.93
              std      466,343.52   183,535.52
              min       -2,597.40    -7,441.50
              25%           25.30       715.62
              50%          282.20     1,930.20
              75%        7,987.90     8,405.60
              max   10,329,553.30 3,588,677.30
kwh_cliente   count     18,687.00     9,188.00
              mean         243.32     1,676.38
              std          717.28     7,224.74
              min            0.00         0.00
              25%            7.96         0.00
              50%           20.20         0.00
              75%           87.00       785.10
              max       14,120.00    83,054.56
soles_cliente count     18,687.00     9,188.00
              mean         256.67     8,965.47
              std          943.67    57,493.27
              min         -772.53    -1,860.37
              25%           12.70       547.38
              50%           24.53     1,082.96
              75%          102.60     2,216.11
              max       23,284.57   828,526.60
precio_kwh    count     16,975.00     3,896.00
              mean           1.66         2.25
              std           16.48        45.48
              min           -1.70         0.00
              25%            0.86         0.79
              50%            1.16         0.97
              75%            1.47         1.24
              max        1,692.73     2,833.08

Medidas de forma: la **asimetría** y la **curtosis** altas indican distribuciones muy sesgadas a la derecha
(pocos registros concentran gran parte del consumo). Por eso conviene trabajar en escala logarítmica.

In [4]:
forma = pd.DataFrame({"asimetria": df[num].skew(), "curtosis": df[num].kurt(),
                      "mediana": df[num].median(), "media": df[num].mean()})
forma

,asimetria,curtosis,mediana,media
clientes,18.96,394.87,2.00,526.44
consumo_kwh,20.15,440.16,129.00,"32,619.29"
monto_soles,18.83,393.66,948.00,"43,249.26"
kwh_cliente,12.21,167.62,17.76,715.68
soles_cliente,19.65,418.45,74.00,"3,127.22"
precio_kwh,94.66,"9,744.31",1.13,1.77


In [5]:
fig = px.histogram(df[df["consumo_kwh"] > 0], x=np.log10(df.loc[df["consumo_kwh"] > 0, "consumo_kwh"]),
                   color="nivel_tension", nbins=60, barmode="overlay", opacity=0.7,
                   labels={"x": "log10(consumo kWh)"}, title="Distribución del consumo por registro (escala log)")
fig.show()

## 4. Composición por tarifa
Se compara la participación de cada tarifa en **clientes**, **energía** y **facturación**.
Un grupo con pocos clientes y mucha facturación revela a los grandes consumidores (media tensión: minería, industria).

In [6]:
tar = df.groupby("tarifa")[["clientes", "consumo_kwh", "monto_soles"]].sum()
part = (tar / tar.sum() * 100).round(2)
part

,clientes,consumo_kwh,monto_soles
tarifa,,,
BT2,0.00,0.00,0.04
BT3,0.00,0.00,0.19
BT4,0.01,0.74,0.60
BT5A,0.00,0.00,0.08
BT5B,99.01,77.32,66.53
BT5D,0.56,6.46,2.54
BT6,0.08,0.40,0.38
MT2,0.09,0.00,9.40
MT3,0.06,0.00,5.99


In [7]:
largo = part.reset_index().melt(id_vars="tarifa", var_name="medida", value_name="porcentaje")
fig = px.bar(largo, x="medida", y="porcentaje", color="tarifa", text_auto=".1f",
             title="Participación por tarifa: clientes vs energía vs facturación (%)")
fig.show()

In [8]:
fig = px.box(df[df["precio_kwh"].between(0, 5)], x="tarifa", y="precio_kwh", points=False,
             title="Precio medio efectivo (S/ por kWh) por tarifa")
fig.show()

## 5. Análisis territorial
### 5.1 Por provincia

In [9]:
prov = (df.groupby("provincia")
          .agg(clientes=("clientes", "sum"), consumo_kwh=("consumo_kwh", "sum"),
               monto_soles=("monto_soles", "sum"), distritos=("ubigeo", "nunique"))
          .assign(kwh_cliente=lambda t: t["consumo_kwh"] / t["clientes"],
                  soles_cliente=lambda t: t["monto_soles"] / t["clientes"])
          .sort_values("clientes", ascending=False))
prov["%clientes"] = prov["clientes"] / prov["clientes"].sum() * 100
prov["%facturacion"] = prov["monto_soles"] / prov["monto_soles"].sum() * 100
prov

,clientes,consumo_kwh,monto_soles,distritos,kwh_cliente,soles_cliente,%clientes,%facturacion
provincia,,,,,,,,
SAN ROMAN,5011427,"371,336,138.25","444,743,851.50",5,74.10,88.75,34.15,36.89
PUNO,3091516,"207,375,008.26","260,657,154.72",15,67.08,84.31,21.07,21.62
AZANGARO,1281743,"37,224,771.26","39,926,656.00",15,29.04,31.15,8.73,3.31
EL COLLAO,908120,"27,519,629.86","32,321,816.10",5,30.30,35.59,6.19,2.68
MELGAR,894140,"41,298,532.39","49,696,814.60",9,46.19,55.58,6.09,4.12
CHUCUITO,778414,"29,516,227.58","30,719,863.80",7,37.92,39.46,5.30,2.55
HUANCANE,647230,"17,660,028.18","19,454,002.02",8,27.29,30.06,4.41,1.61
SAN ANTONIO DE PUTINA,539568,"67,667,129.22","125,762,510.40",5,125.41,233.08,3.68,10.43
LAMPA,437946,"14,786,902.25","66,594,600.20",10,33.76,152.06,2.98,5.52


In [10]:
fig = px.treemap(df, path=[px.Constant("Puno"), "provincia", "distrito"], values="clientes",
                 color="kwh_cliente", color_continuous_scale="YlOrRd", range_color=(0, 150),
                 title="Clientes por provincia y distrito (color = kWh por cliente)")
fig.update_traces(root_color="lightgrey")
fig.show()

### 5.2 Concentración del consumo (Pareto y Lorenz)
El **coeficiente de Gini** mide qué tan desigual es la distribución del consumo entre distritos
(0 = todos consumen igual, 1 = un solo distrito concentra todo).

In [11]:
def lorenz_gini(valores):
    v = np.sort(np.asarray(valores, dtype=float))
    acum = np.insert(np.cumsum(v) / v.sum(), 0, 0)
    pob = np.linspace(0, 1, len(acum))
    gini = 1 - np.sum(np.diff(pob) * (acum[1:] + acum[:-1]))   # 1 - 2·(área bajo la curva)
    return pob, acum, gini

dist_tot = df.groupby(["ubigeo", "distrito"])[["consumo_kwh", "monto_soles", "clientes"]].sum()
fig = go.Figure()
for col, nombre in [("consumo_kwh", "Energía"), ("monto_soles", "Facturación"), ("clientes", "Clientes")]:
    x, y, g = lorenz_gini(dist_tot[col])
    fig.add_scatter(x=x, y=y, name=f"{nombre} (Gini = {g:.2f})")
fig.add_scatter(x=[0, 1], y=[0, 1], name="Igualdad perfecta", line=dict(dash="dash", color="grey"))
fig.update_layout(title="Curva de Lorenz entre distritos", xaxis_title="% acumulado de distritos",
                  yaxis_title="% acumulado", xaxis_tickformat=".0%", yaxis_tickformat=".0%")
fig.show()

In [12]:
pareto = dist_tot.sort_values("monto_soles", ascending=False)
pareto["%acum_facturacion"] = pareto["monto_soles"].cumsum() / pareto["monto_soles"].sum() * 100
n80 = (pareto["%acum_facturacion"] <= 80).sum() + 1
print(f"{n80} de {len(pareto)} distritos ({n80/len(pareto):.0%}) concentran el 80% de la facturación.")
pareto.head(10)

10 de 109 distritos (9%) concentran el 80% de la facturación.


,,consumo_kwh,monto_soles,clientes,%acum_facturacion
ubigeo,distrito,,,,
211101,JULIACA,"354,930,539.57","424,150,070.00",4595150,35.18
210101,PUNO,"183,904,605.12","233,676,813.02",2350618,54.57
211002,ANANEA,"58,177,781.09","106,380,152.90",219171,63.39
210301,MACUSANI,"20,104,193.94","69,787,693.90",650,69.18
210709,SANTA LUCIA,"1,159,120.76","32,663,745.30",17836,71.89
210501,ILAVE,"23,667,700.47","28,021,882.90",809787,74.21
210201,AZANGARO,"22,850,608.39","21,590,577.90",464863,76.00
210801,AYAVIRI,"16,534,065.40","19,397,725.40",381848,77.61
210802,ANTAUTA,"15,492,762.56","19,235,077.50",58133,79.21


## 6. Relaciones entre variables

In [13]:
corr_p = df[num].corr(method="pearson")
corr_s = df[num].corr(method="spearman")
fig = px.imshow(corr_s, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Correlación de Spearman (robusta a la asimetría)")
fig.show()

In [14]:
b5 = df[(df["tarifa"] == "BT5B") & (df["consumo_kwh"] > 0)]
fig = px.scatter(b5, x="clientes", y="consumo_kwh", color="provincia", log_x=True, log_y=True,
                 opacity=0.5, trendline="ols", trendline_scope="overall",
                 trendline_options=dict(log_x=True, log_y=True),
                 title="BT5B: clientes vs consumo por registro (escala log-log)")
fig.show()
res = px.get_trendline_results(fig).iloc[0]["px_fit_results"]
print(f"Pendiente log-log ≈ {res.params[1]:.2f} (≈1 indica consumo proporcional al número de clientes)")

Pendiente log-log ≈ 1.01 (≈1 indica consumo proporcional al número de clientes)


## 7. Indicadores complementarios por distrito
Se crean indicadores nuevos que no están en los datos originales:

| Indicador | Definición |
|---|---|
| `kwh_cliente_bt5b` | kWh por cliente en la tarifa residencial BT5B (consumo típico de un hogar) |
| `precio_kwh_bt5b` | S/ por kWh en la tarifa residencial BT5B |
| `pct_fose` | % de clientes BT5B en grupos con consumo ≤ 100 kWh/mes (rango del FOSE) |
| `pct_bajo_consumo` | % de clientes BT5B con consumo ≤ 30 kWh/mes (posible vulnerabilidad energética) |
| `pct_sin_consumo` | % de clientes en registros con consumo 0 |
| `pct_mt_facturacion` | % de la facturación que proviene de media tensión (presencia de industria/minería) |
| `densidad_clientes` | clientes por km² |

Para varios periodos, los totales se expresan como **promedio mensual**.

*Nota:* como cada fila agrupa clientes, los porcentajes por rango de consumo usan el consumo medio del grupo
(son una aproximación de la distribución real por cliente).

In [15]:
def indicadores(d: pd.DataFrame, by: list) -> pd.DataFrame:
    n = d["periodo_cod"].nunique()
    d = d.assign(_bt5b=d["tarifa"].eq("BT5B"), _bt=d["nivel_tension"].eq("BT"), _mt=d["nivel_tension"].eq("MT"))
    d = d.assign(cli_b5=d["clientes"] * d["_bt5b"], kwh_b5=d["consumo_kwh"] * d["_bt5b"],
                 soles_b5=d["monto_soles"] * d["_bt5b"],
                 cli_bt=d["clientes"] * d["_bt"], kwh_bt=d["consumo_kwh"] * d["_bt"],
                 soles_mt=d["monto_soles"] * d["_mt"],
                 cli_fose=d["clientes"] * (d["_bt5b"] & (d["kwh_cliente"] <= 100)),
                 cli_bajo=d["clientes"] * (d["_bt5b"] & (d["kwh_cliente"] <= 30)),
                 cli_cero=d["clientes"] * d["sin_consumo"])
    g = d.groupby(by).agg(clientes=("clientes", "sum"), consumo_kwh=("consumo_kwh", "sum"),
                          monto_soles=("monto_soles", "sum"), cli_b5=("cli_b5", "sum"),
                          kwh_b5=("kwh_b5", "sum"), soles_b5=("soles_b5", "sum"),
                          cli_bt=("cli_bt", "sum"), kwh_bt=("kwh_bt", "sum"), soles_mt=("soles_mt", "sum"),
                          cli_fose=("cli_fose", "sum"), cli_bajo=("cli_bajo", "sum"), cli_cero=("cli_cero", "sum"))
    out = pd.DataFrame(index=g.index)
    out["clientes"] = g["clientes"] / n
    out["consumo_kwh"] = g["consumo_kwh"] / n
    out["monto_soles"] = g["monto_soles"] / n
    out["kwh_cliente_bt5b"] = g["kwh_b5"] / g["cli_b5"]
    out["kwh_cliente_bt"] = g["kwh_bt"] / g["cli_bt"]
    out["soles_cliente"] = g["monto_soles"] / g["clientes"]
    out["precio_kwh_bt5b"] = g["soles_b5"] / g["kwh_b5"].replace(0, np.nan)
    out["pct_fose"] = g["cli_fose"] / g["cli_b5"] * 100
    out["pct_bajo_consumo"] = g["cli_bajo"] / g["cli_b5"] * 100
    out["pct_sin_consumo"] = g["cli_cero"] / g["clientes"] * 100
    out["pct_mt_facturacion"] = g["soles_mt"] / g["monto_soles"] * 100
    return out.reset_index()

dim = (df.groupby(["ubigeo", "geo_id", "provincia", "distrito"], as_index=False)
         .agg(lat=("lat", "first"), lon=("lon", "first"), area_km2=("area_km2", "first")))
ind = dim.merge(indicadores(df, ["ubigeo"]), on="ubigeo")
ind["densidad_clientes"] = ind["clientes"] / ind["area_km2"]
ind.sort_values("clientes", ascending=False).head(10)

,ubigeo,geo_id,provincia,distrito,lat,lon,area_km2,clientes,consumo_kwh,monto_soles,kwh_cliente_bt5b,kwh_cliente_bt,soles_cliente,precio_kwh_bt5b,pct_fose,pct_bajo_consumo,pct_sin_consumo,pct_mt_facturacion,densidad_clientes
88,211101,211101,SAN ROMAN,JULIACA,-15.45,-70.19,533.99,"109,408.33","8,450,727.13","10,098,811.19",72.03,72.82,92.30,1.12,100.00,0.01,0.11,11.56,204.89
0,210101,210101,PUNO,PUNO,-15.90,-70.06,462.89,"55,967.10","4,378,681.07","5,563,733.64",67.79,70.64,99.41,1.13,100.00,1.24,0.18,19.44,120.91
47,210501,210501,EL COLLAO,ILAVE,-16.23,-69.67,924.37,"19,280.64","563,516.68","667,187.69",24.37,26.56,34.60,1.18,100.00,99.93,0.13,14.79,20.86
15,210201,210201,AZANGARO,AZANGARO,-14.88,-70.15,724.22,"11,068.17","544,062.10","514,061.38",31.71,45.08,46.45,1.10,100.00,31.52,0.05,13.56,15.28
70,210801,210801,MELGAR,AYAVIRI,-14.93,-70.61,"1,021.73","9,091.62","393,668.22","461,850.60",38.39,39.18,50.80,1.11,99.98,0.03,0.10,14.60,8.90
52,210601,210601,HUANCANE,HUANCANE,-15.18,-69.81,389.85,"8,956.31","280,267.92","292,952.42",20.81,30.39,32.71,1.24,99.88,99.78,0.09,3.88,22.97
40,210401,210401,CHUCUITO,JULI,-16.33,-69.50,778.29,"6,836.90","198,873.20","223,000.16",22.92,27.42,32.62,1.23,100.00,100.00,0.03,7.47,8.78
102,211301,211301,YUNGUYO,YUNGUYO,-16.28,-69.09,176.31,"5,719.12","277,090.00","292,937.74",37.05,41.34,51.22,1.11,100.00,0.00,0.12,16.50,32.44
41,210402,210402,CHUCUITO,DESAGUADERO,-16.62,-69.08,176.15,"5,576.67","377,686.73","353,478.70",42.98,58.13,63.39,1.15,100.00,0.04,0.14,14.46,31.66
84,211002,211002,SAN ANTONIO DE PUTINA,ANANEA,-14.69,-69.51,972.30,"5,218.36","1,385,185.26","2,532,860.78",116.23,120.88,485.38,1.18,16.00,2.14,1.28,71.78,5.37


In [16]:
ind[["kwh_cliente_bt5b", "precio_kwh_bt5b", "pct_fose", "pct_bajo_consumo",
     "pct_sin_consumo", "pct_mt_facturacion", "densidad_clientes"]].describe()

,kwh_cliente_bt5b,precio_kwh_bt5b,pct_fose,pct_bajo_consumo,pct_sin_consumo,pct_mt_facturacion,densidad_clientes
count,109.00,109.00,109.00,109.00,109.00,109.00,109.00
mean,31.02,1.45,96.17,76.83,0.87,21.03,9.27
std,52.11,1.11,16.34,39.48,3.65,24.97,25.43
min,0.47,0.92,2.25,0.00,0.00,0.00,0.01
25%,12.30,1.16,99.98,75.70,0.09,4.21,0.90
50%,19.03,1.26,100.00,99.84,0.18,13.32,2.69
75%,26.30,1.44,100.00,100.00,0.37,24.67,7.11
max,461.89,12.37,100.00,100.00,34.59,99.95,204.89


In [17]:
fig = px.choropleth_map(ind, geojson=geo, locations="geo_id", featureidkey="properties.IDDIST",
                        color="kwh_cliente_bt5b", hover_name="distrito",
                        hover_data={"provincia": True, "clientes": ":,.0f", "pct_fose": ":.1f", "geo_id": False},
                        color_continuous_scale="YlOrRd", map_style="carto-positron",
                        center={"lat": -15.0, "lon": -70.0}, zoom=6, opacity=0.8, height=600,
                        title="kWh por cliente residencial (BT5B) por distrito")
fig.show()

## 8. Clustering de distritos (K-Means)
Se agrupan los distritos según su perfil de consumo. Variables (estandarizadas):
`log(clientes)`, `kwh_cliente_bt5b`, `pct_fose`, `pct_sin_consumo`, `pct_mt_facturacion` y `log(densidad)`.
El número de grupos *k* se elige con el **coeficiente de silueta** y el **método del codo**.

In [18]:
feat = ["log_clientes", "kwh_cliente_bt5b", "pct_fose", "pct_sin_consumo", "pct_mt_facturacion", "log_densidad"]
X = ind.assign(log_clientes=np.log1p(ind["clientes"]), log_densidad=np.log1p(ind["densidad_clientes"]))[feat]
X = X.fillna(X.median())
Xs = StandardScaler().fit_transform(X)

ks = range(2, 9)
evals = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(Xs)
    evals.append({"k": k, "inercia": km.inertia_, "silueta": silhouette_score(Xs, km.labels_)})
evals = pd.DataFrame(evals)
fig = px.line(evals, x="k", y=["inercia", "silueta"], markers=True, facet_row="variable",
              title="Selección de k: codo (inercia) y silueta")
fig.update_yaxes(matches=None)
fig.show()
evals

,k,inercia,silueta
0,2,465.60,0.67
1,3,372.35,0.68
2,4,276.28,0.27
3,5,209.47,0.37
4,6,173.55,0.32
5,7,138.46,0.33
6,8,123.01,0.28


In [19]:
k_opt = int(evals.loc[evals["k"].between(3, 6), :].sort_values("silueta", ascending=False).iloc[0]["k"])
km = KMeans(n_clusters=k_opt, n_init=20, random_state=42).fit(Xs)
ind["cluster_id"] = km.labels_

# Nombre descriptivo: los centroides están en unidades estandarizadas (z), así que cada segmento
# se nombra con las 2 variables donde más se aleja del promedio (+ = por encima, − = por debajo).
ETIQUETAS = {
    "log_clientes": ("muchos clientes", "pocos clientes"),
    "kwh_cliente_bt5b": ("alto consumo por cliente", "bajo consumo por cliente"),
    "pct_fose": ("mayoría en rango FOSE", "pocos clientes FOSE"),
    "pct_sin_consumo": ("muchos suministros sin consumo", "pocos suministros sin consumo"),
    "pct_mt_facturacion": ("peso de media tensión (industria/minería)", "poca media tensión"),
    "log_densidad": ("alta densidad (urbano)", "baja densidad (rural)"),
}
centros = pd.DataFrame(km.cluster_centers_, columns=feat)
def nombrar(z: pd.Series) -> str:
    top = z.abs().sort_values(ascending=False).index[:2]
    partes = [ETIQUETAS[f][0] if z[f] > 0 else ETIQUETAS[f][1] for f in top]
    return partes[0][0].upper() + partes[0][1:] + " · " + partes[1]
nombres = pd.Series({i: f"S{i+1}: {nombrar(centros.loc[i])}" for i in centros.index})
display(centros.round(2).set_index(nombres))
ind["cluster"] = ind["cluster_id"].map(nombres)
print("k elegido:", k_opt)
ind.groupby("cluster")[feat[1:5] + ["clientes", "densidad_clientes"]].agg(["mean"]).round(1).assign(
    n_distritos=ind.groupby("cluster").size())

,log_clientes,kwh_cliente_bt5b,pct_fose,pct_sin_consumo,pct_mt_facturacion,log_densidad
S1: Mayoría en rango FOSE · bajo consumo por cliente,0.08,-0.17,0.20,-0.12,-0.07,0.04
S2: Pocos clientes FOSE · alto consumo por cliente,-1.07,3.57,-4.23,0.67,0.90,-0.56
S3: Muchos suministros sin consumo · peso de media tensión (industria/minería),-2.74,-0.59,0.24,9.27,3.00,-1.34


k elegido: 3


,kwh_cliente_bt5b,pct_fose,pct_sin_consumo,pct_mt_facturacion,clientes,densidad_clientes,n_distritos
,mean,mean,mean,mean,mean,mean,
cluster,,,,,,,
S1: Mayoría en rango FOSE · bajo consumo por cliente,22.30,99.50,0.40,19.20,"3,327.40",9.70,103
S2: Pocos clientes FOSE · alto consumo por cliente,216.30,27.30,3.30,43.30,"1,331.80",2.00,5
S3: Muchos suministros sin consumo · peso de media tensión (industria/minería),0.50,100.00,34.60,95.70,8.20,0.00,1


In [20]:
pcs = PCA(n_components=2).fit(Xs)
coords = pcs.transform(Xs)
fig = px.scatter(ind.assign(PC1=coords[:, 0], PC2=coords[:, 1]), x="PC1", y="PC2", color="cluster",
                 size="clientes", hover_name="distrito", hover_data=["provincia"],
                 title=f"Segmentos de distritos en el plano PCA ({pcs.explained_variance_ratio_.sum():.0%} de varianza)")
fig.show()

In [21]:
fig = px.choropleth_map(ind, geojson=geo, locations="geo_id", featureidkey="properties.IDDIST",
                        color="cluster", hover_name="distrito", map_style="carto-positron",
                        center={"lat": -15.0, "lon": -70.0}, zoom=6, opacity=0.75, height=600,
                        title="Segmentación de distritos (K-Means)")
fig.show()

## 9. Dimensión temporal
Si se cargaron varios meses en `data/raw/`, se analiza la evolución mensual por provincia.

In [22]:
if n_periodos > 1:
    ts = (df.groupby(["periodo", "provincia"], as_index=False)[["clientes", "consumo_kwh", "monto_soles"]].sum())
    fig = px.line(ts, x="periodo", y="consumo_kwh", color="provincia", markers=True,
                  title="Evolución mensual del consumo por provincia")
    fig.show()
    tot = df.groupby("periodo")[["clientes", "consumo_kwh", "monto_soles"]].sum()
    display(tot.assign(var_consumo_pct=tot["consumo_kwh"].pct_change() * 100))
else:
    print("Solo hay un periodo cargado (", df["periodo_cod"].iloc[0], "). "
          "Agrega más CSV mensuales en data/raw/ y vuelve a ejecutar los notebooks 1 y 2.")

,clientes,consumo_kwh,monto_soles,var_consumo_pct
periodo,,,,
2023-01-01,336735,"19,738,537.44","25,946,216.70",NaN
2023-02-01,336783,"18,285,801.85","23,283,454.40",-7.36
2023-03-01,336987,"17,806,276.02","21,899,875.00",-2.62
2023-04-01,337449,"20,549,178.14","24,646,463.60",15.40
2023-05-01,338473,"20,034,634.73","24,074,982.90",-2.50
2023-06-01,339870,"21,139,995.18","26,287,241.40",5.52
2023-07-01,340632,"20,881,936.36","25,913,650.40",-1.22
2023-08-01,341197,"21,200,020.10","25,327,893.20",1.52
2023-09-01,341641,"21,551,889.00","25,786,717.50",1.66


## 10. Hallazgos y guardado

In [23]:
top = ind.sort_values("monto_soles", ascending=False).iloc[0]
print(f"• El distrito con mayor facturación es {top['distrito']} ({top['provincia']}), "
      f"con S/ {top['monto_soles']:,.0f} al mes.")
print(f"• {n80} distritos concentran el 80% de la facturación: la demanda está muy concentrada.")
print(f"• En la tarifa residencial BT5B, en promedio el {ind['pct_fose'].mean():.0f}% de los clientes por distrito "
      f"consume ≤ 100 kWh/mes (rango del FOSE).")
print(f"• Se identificaron {k_opt} segmentos de distritos con perfiles de consumo distintos.")

ind.to_csv(OUT / "indicadores_distrito.csv", index=False, encoding="utf-8-sig")
indicadores(df, ["periodo_cod", "provincia"]).to_csv(OUT / "indicadores_provincia.csv", index=False, encoding="utf-8-sig")
print("Guardado: indicadores_distrito.csv, indicadores_provincia.csv")

• El distrito con mayor facturación es JULIACA (SAN ROMAN), con S/ 10,098,811 al mes.
• 10 distritos concentran el 80% de la facturación: la demanda está muy concentrada.
• En la tarifa residencial BT5B, en promedio el 96% de los clientes por distrito consume ≤ 100 kWh/mes (rango del FOSE).
• Se identificaron 3 segmentos de distritos con perfiles de consumo distintos.
Guardado: indicadores_distrito.csv, indicadores_provincia.csv


## 11. KPIs para la Gerencia de Electro Puno
El EDA permite pasar de "muchos números" a **12 indicadores clave** que un gerente puede revisar cada mes.
Se agrupan en 4 bloques, cada uno responde a una pregunta de negocio:

| Bloque | Pregunta gerencial | KPIs |
|---|---|---|
| **Comercial** | ¿Cuánto vendemos y a qué precio? | Facturación mensual · Energía facturada · Precio medio en BT |
| **Clientes y consumo** | ¿A quién atendemos y cuánto consume? | Clientes atendidos · Ingreso medio por cliente · Consumo medio residencial |
| **Riesgo y concentración** | ¿Qué tan vulnerable es el ingreso? | Dependencia de grandes clientes (MT) · Concentración en 10 distritos · Suministros sin consumo |
| **Control y social** | ¿Dónde revisar y a quién proteger? | Facturación bajo revisión · Hogares en rango FOSE · Hogares de muy bajo consumo |

Las definiciones y fórmulas están en `kpis.py`, el mismo módulo que usa el dashboard
(así el notebook y el tablero muestran los mismos valores).
*Facturación bajo revisión* depende de las alertas del Notebook 3, por eso aquí aparece vacía.

> Con datos abiertos no se pueden medir pérdidas de energía, morosidad ni calidad de servicio (SAIDI/SAIFI):
> requieren información interna de la empresa.

In [24]:
import kpis

ficha = kpis.tabla_kpis(df)
ficha["valor"] = [format(v, kpis.POR_ID[k["id"]]["fmt"]) if pd.notna(v) else "(Notebook 3)"
                  for v, k in zip(ficha["valor"], kpis.CATALOGO)]
ficha[["bloque", "kpi", "valor", "unidad", "formula", "pregunta_gerencial"]]

,bloque,kpi,valor,unidad,formula,pregunta_gerencial
0,Comercial,Facturación mensual,"34,968,838",S/ por mes,Σ M_Total / n° de meses,¿Cuánto factura la empresa al mes?
1,Comercial,Energía facturada,"24,772.8",MWh/mes,Σ CEA / 1000 / n° de meses,¿Cuánta energía vendemos?
2,Comercial,Precio medio en BT,1.100,S/ por kWh,Σ M_Total BT / Σ CEA BT (solo registros con co...,¿Cuánto se cobra en promedio por cada kWh en b...
3,Clientes y consumo,Clientes atendidos,"366,818",suministros,Σ Clientes / n° de meses,¿A cuántos suministros atendemos?
4,Clientes y consumo,Ingreso medio por cliente,95.33,S/ por mes,Facturación / Clientes,¿Cuánto aporta en promedio cada cliente?
5,Clientes y consumo,Consumo medio residencial,52.9,kWh/cliente BT5B,Σ CEA BT5B / Σ Clientes BT5B,¿Cuánto consume un hogar típico?
6,Riesgo y concentración,Dependencia de grandes clientes,33.7,% de la facturación en MT,Σ M_Total media tensión / Σ M_Total × 100,¿Qué parte del ingreso depende de minería/indu...
7,Riesgo y concentración,Concentración territorial,80.6,% de la facturación en 10 distritos,Facturación de los 10 distritos mayores / Fact...,¿Qué tan concentrado está el negocio en pocos ...
8,Riesgo y concentración,Suministros sin consumo,0.2,% de clientes,Clientes en registros con CEA = 0 / Clientes ×...,¿Cuántos suministros no registran consumo (ina...
9,Control y social,Facturación bajo revisión,(Notebook 3),S/ en registros con alerta,Σ M_Total de registros marcados como anómalos ...,¿Cuánto dinero está en registros con consumo o...


### 11.1 KPIs por provincia
El gerente también necesita saber **dónde** está cada resultado. Se calculan los KPIs principales por provincia.

In [25]:
kpi_prov = pd.DataFrame({p: kpis.calcular_kpis(g) for p, g in df.groupby("provincia")}).T
kpi_prov = kpi_prov[["facturacion", "clientes", "ingreso_cliente", "kwh_residencial", "pct_mt", "pct_fose"]]
kpi_prov.columns = ["Facturación S/", "Clientes", "S/ por cliente", "kWh/cliente BT5B", "% fact. MT", "% FOSE"]
kpi_prov.sort_values("Facturación S/", ascending=False).style.format("{:,.1f}").background_gradient(cmap="Blues")

,Facturación S/,Clientes,S/ por cliente,kWh/cliente BT5B,% fact. MT,% FOSE
SAN ROMAN,"10,589,139.3","119,319.7",88.7,69.0,11.8,100.0
PUNO,"6,206,122.7","73,607.5",84.3,55.7,21.5,100.0
SAN ANTONIO DE PUTINA,"2,994,345.5","12,846.9",233.1,60.6,69.2,66.2
CARABAYA,"2,187,339.2","6,966.0",314.0,38.7,78.4,99.6
LAMPA,"1,585,585.7","10,427.3",152.1,21.9,80.0,100.0
MELGAR,"1,183,257.5","21,289.0",55.6,39.0,16.5,93.6
AZANGARO,"950,634.7","30,517.7",31.2,20.3,15.5,100.0
EL COLLAO,"769,567.1","21,621.9",35.6,23.5,17.1,100.0
CHUCUITO,"731,425.3","18,533.7",39.5,25.9,12.3,100.0
HUANCANE,"463,190.5","15,410.2",30.1,18.5,10.2,99.9
